# Analysis of Cysteine Environment

In [2]:
import numpy as np
import pandas as pd
import requests
import json
import shutil
import pickle
import io
from pathlib import Path
from Bio import PDB

## 1 - Import data

Input file should be located in the /res subfolder and have one column with accession number of the proteins and another column with position of the cysteines.

In [3]:
filename = "MS-data"
accession_column = "protein accession number"
cysteine_column = "cysteine site"

input_data = pd.read_excel(f"res/{filename}.xlsx")
print(f"{len(input_data)} cysteines in {len(list(np.unique(input_data[accession_column])))} proteins")
input_data

1724 cysteines in 1166 proteins


,protein accession number,gene name,cysteine site
0,P49588,AARS1,C773
1,P49588,AARS1,C903
2,P33527,ABCC1,C744
3,Q8NE71,ABCF1,C655
4,Q9NUJ1,ABHD10,C100
...,...,...,...
1719,Q96KR1,ZFR,C922
1720,Q8N5A5,ZGPAT,C377
1721,Q8TF68,ZNF384,C435
1722,Q2TB10,ZNF800,C359


## 2 -  Create a chunked list of proteins and download new data from AIUPRED and EBI
Needs to run just once, then all the data can be easily retrieved from pickle.<br>

### 2.1 - Create a list of new unique accessions
Avoids downloading twice when two cysteines are in the same protein.

In [4]:
new_accessions = list(accession.split(";")[0] for accession in np.unique(input_data[accession_column]))

### 2.2 - Retrieve old outputs (if present) from pickle and make a list of old accessions

In [5]:
if Path("res/AIUPred_proteins.pkl").exists() and Path("res/EBI_proteins.pkl").exists():
    # Create copies of the old pickle files
    shutil.copy("res/AIUPred_proteins.pkl", "res/AIUPred_proteins_old.pkl")
    shutil.copy("res/EBI_proteins.pkl", "res/EBI_proteins_old.pkl")

    with open("res/AIUPred_proteins_old.pkl", "rb") as f:
        AIUPred_proteins_old = pickle.load(f)
    with open("res/EBI_proteins_old.pkl", "rb") as f:
        EBI_proteins_old = pickle.load(f)
else:
    AIUPred_proteins_old = pd.DataFrame(columns=["accession"])
    EBI_proteins_old = pd.DataFrame(columns=["accession"])

old_accessions_a = list(np.unique(AIUPred_proteins_old["accession"]))
old_accessions_e = list(np.unique(EBI_proteins_old["accession"]))
print(f"Old pickles have same number of accessions: {old_accessions_a == old_accessions_e}")
print(f"{len(old_accessions_a)} proteins in old pickles")

Old pickles have same number of accessions: True
0 proteins in old pickles


### 2.3 - Combine new and old accessions in a comprehensive new list

In [6]:
all_accessions = sorted(list(set(new_accessions + old_accessions_a)), key=str.lower)
print(f"{len(all_accessions)} accessions in the combined list")
all_accessions

1166 accessions in the combined list


['A0AVT1',
 'A6NDU8',
 'A6NHR9',
 'A6NKF1',
 'B2RTY4',
 'O00116',
 'O00151',
 'O00159',
 'O00170',
 'O00186',
 'O00231',
 'O00233',
 'O00267',
 'O00299',
 'O00303',
 'O00400',
 'O00410',
 'O00425',
 'O00429',
 'O00462',
 'O00469',
 'O00471',
 'O00541',
 'O00567',
 'O00571',
 'O00629',
 'O00743',
 'O14524',
 'O14530',
 'O14617',
 'O14682',
 'O14744',
 'O14787',
 'O14818',
 'O14867',
 'O14879',
 'O14929',
 'O14950',
 'O14964',
 'O14967',
 'O14974',
 'O14979',
 'O14981',
 'O15056',
 'O15067',
 'O15078',
 'O15131',
 'O15144',
 'O15228',
 'O15269',
 'O15357',
 'O15371',
 'O15400',
 'O15498',
 'O43143',
 'O43242',
 'O43299',
 'O43447',
 'O43592',
 'O43660',
 'O43663',
 'O43707',
 'O43747',
 'O43765',
 'O43795',
 'O43837',
 'O60216',
 'O60231',
 'O60244',
 'O60292',
 'O60306',
 'O60343',
 'O60508',
 'O60610',
 'O60664',
 'O60701',
 'O60716',
 'O60826',
 'O60832',
 'O60884',
 'O60890',
 'O60936',
 'O75083',
 'O75116',
 'O75122',
 'O75131',
 'O75147',
 'O75165',
 'O75179',
 'O75326',
 'O75347',

### 2.4 -  Divide accessions into chunks
Smaller chunks make it possible to download one bit at a time and save along the way.

In [7]:
chunk_size = 100

accessions = [all_accessions[i : i+chunk_size] for i in range(0, len(all_accessions), chunk_size)]
print(f"Number of chunks: {len(accessions)}")

Number of chunks: 12


###  2.5 - Download redox and disorder data, save pickle chunk files
Takes about 45 minutes for 1000 new proteins.<br>
Retrieve data that are already in the old pickle, saving the time to download them again.<br>
Logging allows following the download as it progresses.

In [8]:
def AIUPred_download(accession, parameters={}):
    ### Access AIUPRED's API and download information
    url = "https://aiupred.elte.hu/rest_api"
    parameters["accession"] = accession 
    try:    # try to download page
        AIUPred_res = session.get(url, params=parameters)
        AIUPred_res.raise_for_status()
    except:
        print(f"Impossible to download AIUPred data for accession {accession}")
        return {"regions": "NaN", "AIUPred": "NaN"}
    return json.loads(AIUPred_res.text)

In [9]:
with requests.Session() as session:
    for i in range(len(accessions)):  # Loop over chunks
    # for i in range(6, len(accessions)):  # in case first 6 chunks already done
        redox = []
        disorder = []

        # Either copy data from old pickle or download disorder and redox data from AIUPred's API
        for accession in accessions[i]:  # Loop over accessions in the chunk
            if accession in AIUPred_proteins_old["accession"].values:   # if accession is already in the old pickle
                redox_old = AIUPred_proteins_old.loc[AIUPred_proteins_old["accession"] == accession, "redox"].iloc[0]
                disorder_old = AIUPred_proteins_old.loc[AIUPred_proteins_old["accession"] == accession, "disorder"].iloc[0]
                redox.append(redox_old)
                disorder.append(disorder_old)
                print(f"AIUPred data for accession {accession} copied from old pickle")
            else:   # if new data need to be downloaded
                # redox
                redox_new = AIUPred_download(accession, {"analysis_type": "redox", "smoothing": "default"})
                redox.append(redox_new["regions"])
                print(f"Redox data for accession {accession} downloaded")
                # disorder
                disorder_new = AIUPred_download(accession)
                disorder.append(disorder_new["AIUPred"])
                print(f"Disorder data for accession {accession} downloaded")

        # Assemble all data for a chunk of accessions in a DataFrame
        AIUPred_chunk = pd.DataFrame({"accession": accessions[i], "redox": redox, "disorder": disorder})
        print(AIUPred_chunk)

        # Save chunk as pickle
        with open(f"res/AIUPred_proteins_chunk_{i+1}.pkl", "wb") as f:
            pickle.dump(AIUPred_chunk, f)
        print(f"AIUPred pickle chunk {i+1} saved\n")

Redox data for accession A0AVT1 downloaded
Disorder data for accession A0AVT1 downloaded
Redox data for accession A6NDU8 downloaded
Disorder data for accession A6NDU8 downloaded
Redox data for accession A6NHR9 downloaded
Disorder data for accession A6NHR9 downloaded
Redox data for accession A6NKF1 downloaded
Disorder data for accession A6NKF1 downloaded
Redox data for accession B2RTY4 downloaded
Disorder data for accession B2RTY4 downloaded
Redox data for accession O00116 downloaded
Disorder data for accession O00116 downloaded
Redox data for accession O00151 downloaded
Disorder data for accession O00151 downloaded
Redox data for accession O00159 downloaded
Disorder data for accession O00159 downloaded
Redox data for accession O00170 downloaded
Disorder data for accession O00170 downloaded
Redox data for accession O00186 downloaded
Disorder data for accession O00186 downloaded
Redox data for accession O00231 downloaded
Disorder data for accession O00231 downloaded
Redox data for access

Disorder data for accession O75351 downloaded
Redox data for accession O75367 downloaded
Disorder data for accession O75367 downloaded
Redox data for accession O75369 downloaded
Disorder data for accession O75369 downloaded
Redox data for accession O75390 downloaded
Disorder data for accession O75390 downloaded
Redox data for accession O75439 downloaded
Disorder data for accession O75439 downloaded
Redox data for accession O75503 downloaded
Disorder data for accession O75503 downloaded
Redox data for accession O75521 downloaded
Disorder data for accession O75521 downloaded
Redox data for accession O75530 downloaded
Disorder data for accession O75530 downloaded
Redox data for accession O75531 downloaded
Disorder data for accession O75531 downloaded
   accession                                              redox  \
0     A0AVT1                                       [[582, 624]]   
1     A6NDU8                                       [[223, 250]]   
2     A6NHR9                             

Redox data for accession P04083 downloaded
Disorder data for accession P04083 downloaded
Redox data for accession P04181 downloaded
Disorder data for accession P04181 downloaded
Redox data for accession P04216 downloaded
Disorder data for accession P04216 downloaded
Redox data for accession P04406 downloaded
Disorder data for accession P04406 downloaded
Redox data for accession P04632 downloaded
Disorder data for accession P04632 downloaded
Redox data for accession P04843 downloaded
Disorder data for accession P04843 downloaded
Redox data for accession P04844 downloaded
Disorder data for accession P04844 downloaded
Redox data for accession P04899 downloaded
Disorder data for accession P04899 downloaded
Redox data for accession P05062 downloaded
Disorder data for accession P05062 downloaded
Redox data for accession P05120 downloaded
Disorder data for accession P05120 downloaded
Redox data for accession P05141 downloaded
Disorder data for accession P05141 downloaded
Redox data for access

Disorder data for accession P13797 downloaded
Redox data for accession P13798 downloaded
Disorder data for accession P13798 downloaded
Redox data for accession P13987 downloaded
Disorder data for accession P13987 downloaded
Redox data for accession P14174 downloaded
Disorder data for accession P14174 downloaded
Redox data for accession P14314 downloaded
Disorder data for accession P14314 downloaded
Redox data for accession P14618 downloaded
Disorder data for accession P14618 downloaded
Redox data for accession P14649 downloaded
Disorder data for accession P14649 downloaded
Redox data for accession P15121 downloaded
Disorder data for accession P15121 downloaded
Redox data for accession P15170 downloaded
Disorder data for accession P15170 downloaded
Redox data for accession P15531 downloaded
Disorder data for accession P15531 downloaded
Redox data for accession P15924 downloaded
Disorder data for accession P15924 downloaded
Redox data for accession P16070 downloaded
Disorder data for acc

Disorder data for accession P26440 downloaded
Redox data for accession P26599 downloaded
Disorder data for accession P26599 downloaded
Redox data for accession P26639 downloaded
Disorder data for accession P26639 downloaded
Redox data for accession P26640 downloaded
Disorder data for accession P26640 downloaded
Redox data for accession P27348 downloaded
Disorder data for accession P27348 downloaded
Redox data for accession P27635 downloaded
Disorder data for accession P27635 downloaded
Redox data for accession P27707 downloaded
Disorder data for accession P27707 downloaded
Redox data for accession P27797 downloaded
Disorder data for accession P27797 downloaded
Redox data for accession P28070 downloaded
Disorder data for accession P28070 downloaded
Redox data for accession P28072 downloaded
Disorder data for accession P28072 downloaded
Redox data for accession P28161 downloaded
Disorder data for accession P28161 downloaded
Redox data for accession P28300 downloaded
Disorder data for acc

...

Redox data for accession Q7Z2Z2 downloaded
Disorder data for accession Q7Z2Z2 downloaded
Redox data for accession Q7Z4Q2 downloaded
Disorder data for accession Q7Z4Q2 downloaded
Redox data for accession Q7Z4V5 downloaded
Disorder data for accession Q7Z4V5 downloaded
Redox data for accession Q7Z6Z7 downloaded
Disorder data for accession Q7Z6Z7 downloaded
Redox data for accession Q86U42 downloaded
Disorder data for accession Q86U42 downloaded
Redox data for accession Q86UP2 downloaded
Disorder data for accession Q86UP2 downloaded
Redox data for accession Q86VI3 downloaded
Disorder data for accession Q86VI3 downloaded
Redox data for accession Q86VP6 downloaded
Disorder data for accession Q86VP6 downloaded
Redox data for accession Q86VU5 downloaded
Disorder data for accession Q86VU5 downloaded
Redox data for accession Q86X55 downloaded
Disorder data for accession Q86X55 downloaded
Redox data for accession Q86XZ4 downloaded
Disorder data for accession Q86XZ4 downloaded
Redox data for access

Disorder data for accession Q93009 downloaded
Redox data for accession Q93034 downloaded
Disorder data for accession Q93034 downloaded
Redox data for accession Q93073 downloaded
Disorder data for accession Q93073 downloaded
Redox data for accession Q969F9 downloaded
Disorder data for accession Q969F9 downloaded
Redox data for accession Q969Q5 downloaded
Disorder data for accession Q969Q5 downloaded
Redox data for accession Q969R2 downloaded
Disorder data for accession Q969R2 downloaded
Redox data for accession Q969V5 downloaded
Disorder data for accession Q969V5 downloaded
Redox data for accession Q96A33 downloaded
Disorder data for accession Q96A33 downloaded
   accession                                       redox  \
0     Q7Z2Z2                                [[945, 971]]   
1     Q7Z4Q2                     [[44, 117], [367, 410]]   
2     Q7Z4V5                                          []   
3     Q7Z6Z7  [[2623, 2682], [3295, 3312], [3325, 3402]]   
4     Q86U42                   

Redox data for accession Q9BSL1 downloaded
Disorder data for accession Q9BSL1 downloaded
Redox data for accession Q9BTT0 downloaded
Disorder data for accession Q9BTT0 downloaded
Redox data for accession Q9BTV5 downloaded
Disorder data for accession Q9BTV5 downloaded
Redox data for accession Q9BTW9 downloaded
Disorder data for accession Q9BTW9 downloaded
Redox data for accession Q9BU76 downloaded
Disorder data for accession Q9BU76 downloaded
Redox data for accession Q9BUF5 downloaded
Disorder data for accession Q9BUF5 downloaded
Redox data for accession Q9BUJ2 downloaded
Disorder data for accession Q9BUJ2 downloaded
Redox data for accession Q9BV81 downloaded
Disorder data for accession Q9BV81 downloaded
Redox data for accession Q9BVP2 downloaded
Disorder data for accession Q9BVP2 downloaded
Redox data for accession Q9BW19 downloaded
Disorder data for accession Q9BW19 downloaded
Redox data for accession Q9BWU0 downloaded
Disorder data for accession Q9BWU0 downloaded
Redox data for access

Redox data for accession Q9NUJ1 downloaded
Disorder data for accession Q9NUJ1 downloaded
Redox data for accession Q9NUP9 downloaded
Disorder data for accession Q9NUP9 downloaded
Redox data for accession Q9NUQ6 downloaded
Disorder data for accession Q9NUQ6 downloaded
Redox data for accession Q9NUQ9 downloaded
Disorder data for accession Q9NUQ9 downloaded
Redox data for accession Q9NVM9 downloaded
Disorder data for accession Q9NVM9 downloaded
Redox data for accession Q9NVN8 downloaded
Disorder data for accession Q9NVN8 downloaded
Redox data for accession Q9NVU7 downloaded
Disorder data for accession Q9NVU7 downloaded
Redox data for accession Q9NVW2 downloaded
Disorder data for accession Q9NVW2 downloaded
Redox data for accession Q9NVZ3 downloaded
Disorder data for accession Q9NVZ3 downloaded
Redox data for accession Q9NW13 downloaded
Disorder data for accession Q9NW13 downloaded
Redox data for accession Q9NW15 downloaded
Disorder data for accession Q9NW15 downloaded
Redox data for access

Redox data for accession Q9Y265 downloaded
Disorder data for accession Q9Y265 downloaded
Redox data for accession Q9Y295 downloaded
Disorder data for accession Q9Y295 downloaded
Redox data for accession Q9Y2L1 downloaded
Disorder data for accession Q9Y2L1 downloaded
Redox data for accession Q9Y2Q5 downloaded
Disorder data for accession Q9Y2Q5 downloaded
Redox data for accession Q9Y2X7 downloaded
Disorder data for accession Q9Y2X7 downloaded
Redox data for accession Q9Y2Z0 downloaded
Disorder data for accession Q9Y2Z0 downloaded
Redox data for accession Q9Y2Z4 downloaded
Disorder data for accession Q9Y2Z4 downloaded
Redox data for accession Q9Y376 downloaded
Disorder data for accession Q9Y376 downloaded
Redox data for accession Q9Y383 downloaded
Disorder data for accession Q9Y383 downloaded
Redox data for accession Q9Y3A5 downloaded
Disorder data for accession Q9Y3A5 downloaded
Redox data for accession Q9Y3B3 downloaded
Disorder data for accession Q9Y3B3 downloaded
Redox data for access

Combine pickle chunks into a single file.

In [10]:
# Retrieve output chunks, merge them, and save as a unique file
AIUPred_proteins = pd.DataFrame()

for i in range(len(accessions)):
    with open(f"res/AIUPred_proteins_chunk_{i+1}.pkl", "rb") as f:
        AIUPred_proteins_chunk = pickle.load(f)
    AIUPred_proteins = pd.concat([AIUPred_proteins, AIUPred_proteins_chunk], ignore_index=True)

with open("res/AIUPred_proteins.pkl", "wb") as f:
    pickle.dump(AIUPred_proteins, f)

AIUPred_proteins

,accession,redox,disorder
0,A0AVT1,"[[582, 624]]","[0.9659407138824463, 0.9646615386009216, 0.968..."
1,A6NDU8,"[[223, 250]]","[0.3372959494590759, 0.32355791330337524, 0.34..."
2,A6NHR9,"[[1868, 1900]]","[0.9728986024856567, 0.9776636362075806, 0.978..."
3,A6NKF1,[],"[0.9777251482009888, 0.9779385328292847, 0.978..."
4,B2RTY4,"[[723, 741], [755, 787], [1170, 1224], [2030, ...","[0.672510027885437, 0.7656236290931702, 0.5662..."
...,...,...,...
1161,Q9Y6E0,[],"[0.5842058658599854, 0.6147359609603882, 0.698..."
1162,Q9Y6G3,"[[11, 72]]","[0.757029116153717, 0.841290295124054, 0.82651..."
1163,Q9Y6M1,[],"[0.2482396364212036, 0.22154109179973602, 0.20..."
1164,Q9Y6M5,"[[414, 462]]","[0.620266318321228, 0.5917750597000122, 0.5621..."


###  2.6 - Download UniProt data from EBI, save pickle chunk files
Takes about 10 minutes for 1000 new proteins.<br>

In [11]:
def EBI_download(accession):
    ### Access EBI's API and download data from there
    url = "https://www.ebi.ac.uk/proteins/api/proteins/"
    requestURL = url+accession 
    try:    # try to download page
        EBI_res = session.get(requestURL, headers={"Accept": "application/json"})
        EBI_res.raise_for_status()
    except:
        print(f"Impossible to download EBI data for accession {accession}")
    if EBI_res.status_code == requests.codes.ok:    # if download worked
        return json.loads(EBI_res.text)

In [12]:
with requests.Session() as session:
    for i in range(len(accessions)):  # Loop over chunks
        EBI_data = []

        # Either copy data from old pickle or download data from EBI's API
        for accession in accessions[i]:  # Loop over accessions in the chunk
            if accession in EBI_proteins_old["accession"].values:   # if accession is already in the old pickle
                EBI_old = EBI_proteins_old.loc[EBI_proteins_old["accession"] == accession, "EBI_data"].iloc[0]
                EBI_data.append(EBI_old)
                print(f"EBI data for accession {accession} copied from old pickle")
            else:   # if new data need to be downloaded
                EBI_new = EBI_download(accession)
                EBI_data.append(EBI_new)
                print(f"EBI data for accession {accession} downloaded")

        # Assemble all data for a chunk of accessions in a DataFrame
        EBI_chunk = pd.DataFrame({"accession": accessions[i], "EBI_data": EBI_data})
        print(EBI_chunk)

        # Save chunk as pickle
        with open(f"res/EBI_proteins_chunk_{i+1}.pkl", "wb") as f:
            pickle.dump(EBI_chunk, f)
        print(f"EBI pickle chunk {i+1} saved\n")

EBI data for accession A0AVT1 downloaded
EBI data for accession A6NDU8 downloaded
EBI data for accession A6NHR9 downloaded
EBI data for accession A6NKF1 downloaded
EBI data for accession B2RTY4 downloaded
EBI data for accession O00116 downloaded
EBI data for accession O00151 downloaded
EBI data for accession O00159 downloaded
EBI data for accession O00170 downloaded
EBI data for accession O00186 downloaded
EBI data for accession O00231 downloaded
EBI data for accession O00233 downloaded
EBI data for accession O00267 downloaded
EBI data for accession O00299 downloaded
EBI data for accession O00303 downloaded
EBI data for accession O00400 downloaded
EBI data for accession O00410 downloaded
EBI data for accession O00425 downloaded
EBI data for accession O00429 downloaded
EBI data for accession O00462 downloaded
EBI data for accession O00469 downloaded
EBI data for accession O00471 downloaded
EBI data for accession O00541 downloaded
EBI data for accession O00567 downloaded
EBI data for acc

EBI data for accession P05556 downloaded
EBI data for accession P06132 downloaded
EBI data for accession P06400 downloaded
EBI data for accession P06733 downloaded
EBI data for accession P06746 downloaded
EBI data for accession P07237 downloaded
EBI data for accession P07355 downloaded
EBI data for accession P07384 downloaded
EBI data for accession P07437 downloaded
EBI data for accession P07602 downloaded
EBI data for accession P07737 downloaded
EBI data for accession P07814 downloaded
EBI data for accession P07858 downloaded
EBI data for accession P07900 downloaded
EBI data for accession P07954 downloaded
EBI data for accession P07992 downloaded
EBI data for accession P08133 downloaded
EBI data for accession P08174 downloaded
EBI data for accession P08237 downloaded
EBI data for accession P08240 downloaded
   accession                                           EBI_data
0     O75533  {'accession': 'O75533', 'id': 'SF3B1_HUMAN', '...
1     O75534  {'accession': 'O75534', 'id': 'CSDE1_H

EBI data for accession P30050 downloaded
EBI data for accession P30084 downloaded
EBI data for accession P30086 downloaded
EBI data for accession P30101 downloaded
EBI data for accession P30153 downloaded
EBI data for accession P30260 downloaded
EBI data for accession P30281 downloaded
EBI data for accession P30519 downloaded
EBI data for accession P30530 downloaded
EBI data for accession P30626 downloaded
EBI data for accession P31040 downloaded
EBI data for accession P31153 downloaded
EBI data for accession P31323 downloaded
EBI data for accession P31327 downloaded
EBI data for accession P31751 downloaded
EBI data for accession P31930 downloaded
EBI data for accession P31939 downloaded
EBI data for accession P31943 downloaded
EBI data for accession P31948 downloaded
EBI data for accession P32969 downloaded
EBI data for accession P33121 downloaded
EBI data for accession P33316 downloaded
EBI data for accession P33527 downloaded
EBI data for accession P33993 downloaded
EBI data for acc

EBI data for accession P56589 downloaded
EBI data for accession P57721 downloaded
EBI data for accession P60174 downloaded
EBI data for accession P60709 downloaded
EBI data for accession P60866 downloaded
EBI data for accession P60891 downloaded
EBI data for accession P60900 downloaded
EBI data for accession P60953 downloaded
EBI data for accession P60981 downloaded
EBI data for accession P61011 downloaded
EBI data for accession P61081 downloaded
EBI data for accession P61086 downloaded
EBI data for accession P61088 downloaded
EBI data for accession P61158 downloaded
EBI data for accession P61160 downloaded
EBI data for accession P61224 downloaded
EBI data for accession P61247 downloaded
EBI data for accession P61964 downloaded
EBI data for accession P61970 downloaded
EBI data for accession P61978 downloaded
EBI data for accession P62072 downloaded
EBI data for accession P62136 downloaded
EBI data for accession P62195 downloaded
EBI data for accession P62241 downloaded
EBI data for acc

EBI data for accession Q14980 downloaded
EBI data for accession Q14C86 downloaded
EBI data for accession Q15008 downloaded
EBI data for accession Q15018 downloaded
EBI data for accession Q15019 downloaded
EBI data for accession Q15020 downloaded
EBI data for accession Q15024 downloaded
EBI data for accession Q15029 downloaded
EBI data for accession Q15041 downloaded
EBI data for accession Q15042 downloaded
EBI data for accession Q15047 downloaded
EBI data for accession Q15056 downloaded
EBI data for accession Q15075 downloaded
EBI data for accession Q15102 downloaded
EBI data for accession Q15147 downloaded
EBI data for accession Q15149 downloaded
EBI data for accession Q15154 downloaded
EBI data for accession Q15181 downloaded
EBI data for accession Q15185 downloaded
EBI data for accession Q15311 downloaded
   accession                                           EBI_data
0     Q08257  {'accession': 'Q08257', 'id': 'QOR_HUMAN', 'pr...
1     Q08AF3  {'accession': 'Q08AF3', 'id': 'SLFN5_H

EBI data for accession Q8NAP3 downloaded
EBI data for accession Q8NAV1 downloaded
EBI data for accession Q8NB16 downloaded
EBI data for accession Q8NB49 downloaded
EBI data for accession Q8NE71 downloaded
EBI data for accession Q8NEN9 downloaded
EBI data for accession Q8NEW0 downloaded
EBI data for accession Q8NF37 downloaded
EBI data for accession Q8NF91 downloaded
EBI data for accession Q8NFH4 downloaded
EBI data for accession Q8NG68 downloaded
EBI data for accession Q8TAE8 downloaded
EBI data for accession Q8TB52 downloaded
EBI data for accession Q8TBC4 downloaded
EBI data for accession Q8TCG1 downloaded
EBI data for accession Q8TCS8 downloaded
EBI data for accession Q8TDD1 downloaded
EBI data for accession Q8TDX7 downloaded
EBI data for accession Q8TEQ6 downloaded
EBI data for accession Q8TEX9 downloaded
EBI data for accession Q8TF42 downloaded
EBI data for accession Q8TF68 downloaded
EBI data for accession Q8WTW4 downloaded
EBI data for accession Q8WUM4 downloaded
EBI data for acc

EBI data for accession Q9H0C8 downloaded
EBI data for accession Q9H1B7 downloaded
EBI data for accession Q9H204 downloaded
EBI data for accession Q9H2G2 downloaded
EBI data for accession Q9H2M9 downloaded
EBI data for accession Q9H2U2 downloaded
EBI data for accession Q9H3U1 downloaded
EBI data for accession Q9H488 downloaded
EBI data for accession Q9H4A3 downloaded
EBI data for accession Q9H4H8 downloaded
EBI data for accession Q9H4L4 downloaded
EBI data for accession Q9H583 downloaded
EBI data for accession Q9H6R4 downloaded
EBI data for accession Q9H7C4 downloaded
EBI data for accession Q9H8Y8 downloaded
EBI data for accession Q9H936 downloaded
EBI data for accession Q9H9A5 downloaded
EBI data for accession Q9HAU4 downloaded
EBI data for accession Q9HAV0 downloaded
EBI data for accession Q9HAV4 downloaded
EBI data for accession Q9HB71 downloaded
EBI data for accession Q9HBH5 downloaded
EBI data for accession Q9HC35 downloaded
EBI data for accession Q9HC36 downloaded
EBI data for acc

Combine pickle chunks into a single file.

In [13]:
# Retrieve output chunks, merge them, and save as a unique file
EBI_proteins = pd.DataFrame()

for i in range(len(accessions)):
    with open(f"res/EBI_proteins_chunk_{i+1}.pkl", "rb") as f:
        EBI_proteins_chunk = pickle.load(f)
    EBI_proteins = pd.concat([EBI_proteins, EBI_proteins_chunk], ignore_index=True)

with open("res/EBI_proteins.pkl", "wb") as f:
    pickle.dump(EBI_proteins, f)

EBI_proteins

,accession,EBI_data
0,A0AVT1,"{'accession': 'A0AVT1', 'id': 'UBA6_HUMAN', 'p..."
1,A6NDU8,"{'accession': 'A6NDU8', 'id': 'RIMC1_HUMAN', '..."
2,A6NHR9,"{'accession': 'A6NHR9', 'id': 'SMHD1_HUMAN', '..."
3,A6NKF1,"{'accession': 'A6NKF1', 'id': 'SAC31_HUMAN', '..."
4,B2RTY4,"{'accession': 'B2RTY4', 'id': 'MYO9A_HUMAN', '..."
...,...,...
1161,Q9Y6E0,"{'accession': 'Q9Y6E0', 'id': 'STK24_HUMAN', '..."
1162,Q9Y6G3,"{'accession': 'Q9Y6G3', 'id': 'RM42_HUMAN', 'p..."
1163,Q9Y6M1,"{'accession': 'Q9Y6M1', 'id': 'IF2B2_HUMAN', '..."
1164,Q9Y6M5,"{'accession': 'Q9Y6M5', 'id': 'ZNT1_HUMAN', 'p..."


## 3 - Analyze data and extract information about each cysteine, including AlphaFold proximity data
### 3.1 - Retrieve AIUPred and EBI outputs from pickle and redistribute protein data over all cysteines

In [14]:
with open("res/AIUPred_proteins.pkl", "rb") as f:
    AIUPred_proteins = pickle.load(f)
with open("res/EBI_proteins.pkl", "rb") as f:
    EBI_proteins = pickle.load(f)

# Distribute protein data over all cysteines
redox = []
disorder = []
EBI_data = []

for i in range(len(input_data)):
    redox.append(AIUPred_proteins.loc[AIUPred_proteins["accession"] == input_data[accession_column][i].split(";")[0], "redox"].iloc[0])
    disorder.append(AIUPred_proteins.loc[AIUPred_proteins["accession"] == input_data[accession_column][i].split(";")[0], "disorder"].iloc[0])
    EBI_data.append(EBI_proteins.loc[EBI_proteins["accession"] == input_data[accession_column][i].split(";")[0], "EBI_data"].iloc[0])

AIUPred_output = pd.concat([input_data, pd.DataFrame({"redox": redox, "disorder": disorder})], axis=1)
AIUPred_output

,protein accession number,gene name,cysteine site,redox,disorder
0,P49588,AARS1,C773,[],"[0.6165097951889038, 0.630347728729248, 0.5685..."
1,P49588,AARS1,C903,[],"[0.6165097951889038, 0.630347728729248, 0.5685..."
2,P33527,ABCC1,C744,[],"[0.8710123300552368, 0.9304897785186768, 0.922..."
3,Q8NE71,ABCF1,C655,[],"[0.974740743637085, 0.9731984734535217, 0.9748..."
4,Q9NUJ1,ABHD10,C100,[],"[0.8191232085227966, 0.8696340918540955, 0.899..."
...,...,...,...,...,...
1719,Q96KR1,ZFR,C922,"[[327, 357], [379, 412], [568, 612]]","[0.7348648309707642, 0.720794141292572, 0.8413..."
1720,Q8N5A5,ZGPAT,C377,"[[317, 385]]","[0.5898325443267822, 0.5299968123435974, 0.569..."
1721,Q8TF68,ZNF384,C435,"[[223, 466]]","[0.9632839560508728, 0.9637404680252075, 0.969..."
1722,Q2TB10,ZNF800,C359,"[[0, 28], [225, 261], [283, 318], [347, 379], ...","[0.5370450615882874, 0.5559547543525696, 0.560..."


In [15]:
EBI_output = pd.concat([input_data, pd.DataFrame({"EBI_data": EBI_data})], axis=1)
EBI_output

,protein accession number,gene name,cysteine site,EBI_data
0,P49588,AARS1,C773,"{'accession': 'P49588', 'id': 'SYAC_HUMAN', 'p..."
1,P49588,AARS1,C903,"{'accession': 'P49588', 'id': 'SYAC_HUMAN', 'p..."
2,P33527,ABCC1,C744,"{'accession': 'P33527', 'id': 'MRP1_HUMAN', 'p..."
3,Q8NE71,ABCF1,C655,"{'accession': 'Q8NE71', 'id': 'ABCF1_HUMAN', '..."
4,Q9NUJ1,ABHD10,C100,"{'accession': 'Q9NUJ1', 'id': 'ABHDA_HUMAN', '..."
...,...,...,...,...
1719,Q96KR1,ZFR,C922,"{'accession': 'Q96KR1', 'id': 'ZFR_HUMAN', 'pr..."
1720,Q8N5A5,ZGPAT,C377,"{'accession': 'Q8N5A5', 'id': 'ZGPAT_HUMAN', '..."
1721,Q8TF68,ZNF384,C435,"{'accession': 'Q8TF68', 'id': 'ZN384_HUMAN', '..."
1722,Q2TB10,ZNF800,C359,"{'accession': 'Q2TB10', 'id': 'ZN800_HUMAN', '..."


### 3.2 - Transform AIUPred output into specific features

In [16]:
sensitive = []
within = []
disorder = []

for i in range(len(AIUPred_output)):
    cysteine = list(AIUPred_output[cysteine_column][i].split(","))[0]  # if several cysteines, consider just the first
    cysteine_pos = int(cysteine[1:] if cysteine[0].lower() == "c" else cysteine)  # if cysteine starts with a "C", remove it
    withinyn = "No"
    # Is there a region that is sensitive to oxidation?
    if AIUPred_output["redox"][i] == "NaN":
        sensitive.append("NaN")
        within.append("NaN")
    else:
        if len(AIUPred_output["redox"][i]) == 0:
            sensitive.append("No")
        else:
            sensitive.append("Yes")
            # Is the cysteine within that region?
            for j in range(len(AIUPred_output["redox"][i])):
                if (AIUPred_output["redox"][i][j][0] <= cysteine_pos) & (cysteine_pos <= AIUPred_output["redox"][i][j][1]):
                    withinyn = "Yes"    # if cysteine is within one of the regions
        within.append(withinyn)
    # What is the disorder value of the cysteine?
    if AIUPred_output["disorder"][i] == "NaN" or cysteine_pos > len(AIUPred_output["disorder"][i]):
        disorder.append("NaN")
    else:
        disorder.append(round(AIUPred_output["disorder"][i][cysteine_pos-1], 2))

AIUPred_features = pd.DataFrame({"sensitive": sensitive, "cysteine_within": within, "disorder_value": disorder})

### 3.3 - Transform EBI output into the desired features

In [17]:
proteinLength = []
fullName = []
subcellularLoc = []

for i in range(len(EBI_output)):
    
    # Protein length and sequence
    proteinLength.append(EBI_output["EBI_data"][i]["sequence"]["length"])
#     protein_sequence = EBI_output["EBI_data"][i]["sequence"]["sequence"]
    
    # Full protein name
    fullName.append(EBI_output["EBI_data"][i]["protein"]["recommendedName"]["fullName"]["value"])
    
    # Subcellular location
    locations = []
    try:
        for j in EBI_output["EBI_data"][i]["comments"]:
            if j["type"] == "SUBCELLULAR_LOCATION":
                try:
                    for k in j["locations"]:
                        locations.append(k["location"]["value"])
                except Exception:
                    pass
    except:
        pass
    subcellularLoc.append(locations)

EBI_features = pd.DataFrame({"protein_length": proteinLength, "full_name": fullName, "subcellular_loc": subcellularLoc})

### 3.4 - Download AlphaFold data and extract proximity residues for each cysteine

In [18]:
class AlphaFoldClient:
    def __init__(self):
        self.session = requests.Session()
        self.parser = PDB.PDBParser()

    def get_pdb(self, accession):
        ### Access AlphaFold's API and get model ID and pdb_url for the "correct" model
        protein_length = EBI_proteins.loc[EBI_proteins["accession"] == accession, "EBI_data"].iloc[0]["sequence"]["length"]
        url = "https://alphafold.ebi.ac.uk/api/prediction/"
        requestURL = url+accession
        global no_output, no_length
        model_ID, pdb_url = "NaN", "NaN"
        try:    # try to download metadata page
            AlphaFold_res = self.session.get(requestURL, params={"include_complexes": "false"})
            AlphaFold_res.raise_for_status()
        except:
            print(f"Impossible to retrieve AlphaFold metadata for accession {accession}")
            no_output+=1
            return model_ID, pdb_url   # if there is no AlphaFold output
        AlphaFold_output = json.loads(AlphaFold_res.text)
        print(f"AlphaFold metadata for accession {accession} downloaded")        
        # Loop through models to select the one where sequence is same length as in UniProt
        for i in AlphaFold_output:
            if i["sequenceEnd"] == protein_length:
                model_ID = i["modelEntityId"]
                pdb_url = i["pdbUrl"]
                return model_ID, pdb_url   # if a correct model is found
        print(f"No AlphaFold model with same sequence length as UniProt for accession {accession}")
        no_length+=1
        return model_ID, pdb_url   # if there is no model that fits the UniProt sequence length

    def get_chain(self, accession):
        ### Access pdb model on AlphaFold's API and extract chain
        model_ID, pdb_url = self.get_pdb(accession)
        try:    # try to download pdb file
            pdb_res = self.session.get(pdb_url)
            pdb_res.raise_for_status()
        except:
            print(f"Impossible to retrieve pdb data for accession {accession}")
            return model_ID, "NaN"
        # Parse the pdb file-like object to extract the chain with info on all residues
        structure = self.parser.get_structure(accession, io.StringIO(pdb_res.text))
        model = structure[0]
        chain = model["A"]
        print(f"AlphaFold chain for accession {accession} downloaded")
        return model_ID, chain

In [19]:
def distance(chain, residue1, residue2):
    ### Calculate distance between C-alpha atoms of two residues
    # Extract the C-alpha (CA) atoms for both residues
    atom1 = chain[residue1]["CA"]
    atom2 = chain[residue2]["CA"]
    
    # Calculate the vector distance
    diff = atom1.get_coord() - atom2.get_coord()
    return np.sqrt(np.sum(diff ** 2))

Loop over all the cysteines, download AlphaFold chain for their protein, then scan the chain and extract residues in proximity to the cysteine.<br>
Combine cysteine information, AlphaFold model ID, and proximity residues data in a DataFrame.<br>
Takes about 2,5 minutes for 1000 cysteines.

In [20]:
proximity = 20   # distance of interest, in Å
no_output = 0
no_length = 0
AlphaFold_model_IDs = []
AlphaFold_proximity_residues = []
old_accession = None
client = AlphaFoldClient()

# Download data from AlphaFold's API
for i in range(len(input_data)):  # Loop over rows
    accession = input_data[accession_column][i].split(";")[0]
    cysteine = list(str(input_data[cysteine_column][i]).split(","))[0]  # if several cysteines, consider just the first
    cysteine_pos = int(cysteine[1:] if cysteine[0].lower() == "c" else cysteine)  # if cysteine starts with a "C", remove it
    proximity_residues = {}

    if accession != old_accession:  # download only if it's a new protein
        model_ID, chain = client.get_chain(accession)

    try:   # Loop over residues and select those close to the cysteine
        for k in range (sum(1 for res in chain)):
            dist = distance(chain, cysteine_pos, k+1)
            if dist <= proximity:
                proximity_residues[k+1] = round(dist, 2)
    except:   # If no chain could be downloaded
        pass

    AlphaFold_model_IDs.append(model_ID)
    AlphaFold_proximity_residues.append(proximity_residues)        
    old_accession = accession

# Assemble all data in a DataFrame
AlphaFold_output = pd.concat([input_data, pd.DataFrame({"AlphaFold_model_ID": AlphaFold_model_IDs,
                                                         "proximity_residues": AlphaFold_proximity_residues})], axis=1)

print(AlphaFold_output)
print(f"\nTotal accessions with no output: {no_output}\nTotal accessions with no model of the correct length: {no_length}\n")

AlphaFold metadata for accession P49588 downloaded
AlphaFold chain for accession P49588 downloaded
AlphaFold metadata for accession P33527 downloaded
AlphaFold chain for accession P33527 downloaded
AlphaFold metadata for accession Q8NE71 downloaded
AlphaFold chain for accession Q8NE71 downloaded
AlphaFold metadata for accession Q9NUJ1 downloaded
AlphaFold chain for accession Q9NUJ1 downloaded
AlphaFold metadata for accession Q8IZP0 downloaded
AlphaFold chain for accession Q8IZP0 downloaded
AlphaFold metadata for accession Q9NYB9 downloaded
AlphaFold chain for accession Q9NYB9 downloaded
AlphaFold metadata for accession Q12979 downloaded
AlphaFold chain for accession Q12979 downloaded
AlphaFold metadata for accession Q15018 downloaded
AlphaFold chain for accession Q15018 downloaded
AlphaFold metadata for accession Q13085 downloaded
AlphaFold chain for accession Q13085 downloaded
AlphaFold metadata for accession P49748 downloaded
AlphaFold chain for accession P49748 downloaded
AlphaFold 

AlphaFold chain for accession P48444 downloaded
AlphaFold metadata for accession P18085 downloaded
AlphaFold chain for accession P18085 downloaded
AlphaFold metadata for accession Q9NP61 downloaded
AlphaFold chain for accession Q9NP61 downloaded
AlphaFold metadata for accession Q13795 downloaded
AlphaFold chain for accession Q13795 downloaded
AlphaFold metadata for accession Q68EM7 downloaded
AlphaFold chain for accession Q68EM7 downloaded
AlphaFold metadata for accession Q8N392 downloaded
AlphaFold chain for accession Q8N392 downloaded
AlphaFold metadata for accession Q52LW3 downloaded
AlphaFold chain for accession Q52LW3 downloaded
AlphaFold metadata for accession Q92888 downloaded
AlphaFold chain for accession Q92888 downloaded
AlphaFold metadata for accession Q92974 downloaded
AlphaFold chain for accession Q92974 downloaded
AlphaFold metadata for accession Q14155 downloaded
AlphaFold chain for accession Q14155 downloaded
AlphaFold metadata for accession Q9Y4X5 downloaded
AlphaFold 

AlphaFold metadata for accession Q13042 downloaded
AlphaFold chain for accession Q13042 downloaded
AlphaFold metadata for accession P30260 downloaded
AlphaFold chain for accession P30260 downloaded
AlphaFold metadata for accession Q16543 downloaded
AlphaFold chain for accession Q16543 downloaded
AlphaFold metadata for accession O60508 downloaded
AlphaFold chain for accession O60508 downloaded
AlphaFold metadata for accession P60953 downloaded
AlphaFold chain for accession P60953 downloaded
AlphaFold metadata for accession Q99459 downloaded
AlphaFold chain for accession Q99459 downloaded
AlphaFold metadata for accession Q99741 downloaded
AlphaFold chain for accession Q99741 downloaded
AlphaFold metadata for accession Q00537 downloaded
AlphaFold chain for accession Q00537 downloaded
AlphaFold metadata for accession P11802 downloaded
AlphaFold chain for accession P11802 downloaded
AlphaFold metadata for accession Q96JB5 downloaded
AlphaFold chain for accession Q96JB5 downloaded
AlphaFold 

AlphaFold chain for accession Q14203 downloaded
AlphaFold metadata for accession Q96GG9 downloaded
AlphaFold chain for accession Q96GG9 downloaded
AlphaFold metadata for accession O94760 downloaded
AlphaFold chain for accession O94760 downloaded
AlphaFold metadata for accession Q16531 downloaded
AlphaFold chain for accession Q16531 downloaded
AlphaFold metadata for accession P39656 downloaded
AlphaFold chain for accession P39656 downloaded
AlphaFold metadata for accession Q92499 downloaded
AlphaFold chain for accession Q92499 downloaded
AlphaFold metadata for accession Q92841 downloaded
AlphaFold chain for accession Q92841 downloaded
AlphaFold metadata for accession Q9GZR7 downloaded
AlphaFold chain for accession Q9GZR7 downloaded
AlphaFold metadata for accession Q13838 downloaded
AlphaFold chain for accession Q13838 downloaded
AlphaFold metadata for accession O00571 downloaded
AlphaFold chain for accession O00571 downloaded
AlphaFold metadata for accession Q7L014 downloaded
AlphaFold 

AlphaFold chain for accession O94919 downloaded
AlphaFold metadata for accession P06733 downloaded
AlphaFold chain for accession P06733 downloaded
AlphaFold metadata for accession P09104 downloaded
AlphaFold chain for accession P09104 downloaded
AlphaFold metadata for accession P29317 downloaded
AlphaFold chain for accession P29317 downloaded
AlphaFold metadata for accession P07814 downloaded
AlphaFold chain for accession P07814 downloaded
AlphaFold metadata for accession P07992 downloaded
AlphaFold chain for accession P07992 downloaded
AlphaFold metadata for accession P84090 downloaded
AlphaFold chain for accession P84090 downloaded
AlphaFold metadata for accession O94905 downloaded
AlphaFold chain for accession O94905 downloaded
AlphaFold metadata for accession Q96HE7 downloaded
AlphaFold chain for accession Q96HE7 downloaded
AlphaFold metadata for accession P10768 downloaded
AlphaFold chain for accession P10768 downloaded
AlphaFold metadata for accession Q9BSJ8 downloaded
AlphaFold 

...

AlphaFold chain for accession P23467 downloaded
AlphaFold metadata for accession Q9Y3E5 downloaded
AlphaFold chain for accession Q9Y3E5 downloaded
AlphaFold metadata for accession Q13610 downloaded
AlphaFold chain for accession Q13610 downloaded
AlphaFold metadata for accession P49023 downloaded
AlphaFold chain for accession P49023 downloaded
AlphaFold metadata for accession P11216 downloaded
AlphaFold chain for accession P11216 downloaded
AlphaFold metadata for accession P47897 downloaded
AlphaFold chain for accession P47897 downloaded
AlphaFold metadata for accession Q9BXR0 downloaded
AlphaFold chain for accession Q9BXR0 downloaded
AlphaFold metadata for accession Q9BXF6 downloaded
AlphaFold chain for accession Q9BXF6 downloaded
AlphaFold metadata for accession Q969Q5 downloaded
AlphaFold chain for accession Q969Q5 downloaded
AlphaFold metadata for accession Q13637 downloaded
AlphaFold chain for accession Q13637 downloaded
AlphaFold metadata for accession P20337 downloaded
AlphaFold 

AlphaFold chain for accession P62241 downloaded
AlphaFold metadata for accession P10301 downloaded
AlphaFold chain for accession P10301 downloaded
AlphaFold metadata for accession Q9P2E9 downloaded
AlphaFold chain for accession Q9P2E9 downloaded
AlphaFold metadata for accession P23921 downloaded
AlphaFold chain for accession P23921 downloaded
AlphaFold metadata for accession Q5JTH9 downloaded
AlphaFold chain for accession Q5JTH9 downloaded
AlphaFold metadata for accession O76021 downloaded
AlphaFold chain for accession O76021 downloaded
AlphaFold metadata for accession Q9NQC3 downloaded
AlphaFold chain for accession Q9NQC3 downloaded
AlphaFold metadata for accession Q9Y265 downloaded
AlphaFold chain for accession Q9Y265 downloaded
AlphaFold metadata for accession A6NKF1 downloaded
AlphaFold chain for accession A6NKF1 downloaded
Impossible to retrieve AlphaFold metadata for accession Q9NZJ4
Impossible to retrieve pdb data for accession Q9NZJ4
AlphaFold metadata for accession Q9UBE0 down

AlphaFold chain for accession P09132 downloaded
AlphaFold metadata for accession P61011 downloaded
AlphaFold chain for accession P61011 downloaded
AlphaFold metadata for accession Q9UHB9 downloaded
AlphaFold chain for accession Q9UHB9 downloaded
AlphaFold metadata for accession O76094 downloaded
AlphaFold chain for accession O76094 downloaded
AlphaFold metadata for accession P49458 downloaded
AlphaFold chain for accession P49458 downloaded
AlphaFold metadata for accession P08240 downloaded
AlphaFold chain for accession P08240 downloaded
AlphaFold metadata for accession Q9Y5M8 downloaded
AlphaFold chain for accession Q9Y5M8 downloaded
AlphaFold metadata for accession Q9BXP5 downloaded
AlphaFold chain for accession Q9BXP5 downloaded
AlphaFold metadata for accession Q07955 downloaded
AlphaFold chain for accession Q07955 downloaded
AlphaFold metadata for accession P84103 downloaded
AlphaFold chain for accession P84103 downloaded
AlphaFold metadata for accession Q08170 downloaded
AlphaFold 

AlphaFold chain for accession Q12931 downloaded
AlphaFold metadata for accession Q8WVT3 downloaded
AlphaFold chain for accession Q8WVT3 downloaded
AlphaFold metadata for accession Q15643 downloaded
AlphaFold chain for accession Q15643 downloaded
AlphaFold metadata for accession Q9UJA5 downloaded
AlphaFold chain for accession Q9UJA5 downloaded
AlphaFold metadata for accession Q96FX7 downloaded
AlphaFold chain for accession Q96FX7 downloaded
AlphaFold metadata for accession P43897 downloaded
AlphaFold chain for accession P43897 downloaded
AlphaFold metadata for accession Q99598 downloaded
AlphaFold chain for accession Q99598 downloaded
AlphaFold metadata for accession Q16762 downloaded
AlphaFold chain for accession Q16762 downloaded
AlphaFold metadata for accession Q5R3I4 downloaded
AlphaFold chain for accession Q5R3I4 downloaded
AlphaFold metadata for accession Q15361 downloaded
AlphaFold chain for accession Q15361 downloaded
AlphaFold metadata for accession Q8NG68 downloaded
AlphaFold 

AlphaFold chain for accession Q9Y2Z4 downloaded
AlphaFold metadata for accession O15498 downloaded
AlphaFold chain for accession O15498 downloaded
AlphaFold metadata for accession Q04917 downloaded
AlphaFold chain for accession Q04917 downloaded
AlphaFold metadata for accession P27348 downloaded
AlphaFold chain for accession P27348 downloaded
AlphaFold metadata for accession P63104 downloaded
AlphaFold chain for accession P63104 downloaded
AlphaFold metadata for accession Q8NAP3 downloaded
AlphaFold chain for accession Q8NAP3 downloaded
AlphaFold metadata for accession Q8IWR0 downloaded
AlphaFold chain for accession Q8IWR0 downloaded
AlphaFold metadata for accession Q7Z2W4 downloaded
AlphaFold chain for accession Q7Z2W4 downloaded
AlphaFold metadata for accession Q96KR1 downloaded
AlphaFold chain for accession Q96KR1 downloaded
AlphaFold metadata for accession Q8N5A5 downloaded
AlphaFold chain for accession Q8N5A5 downloaded
AlphaFold metadata for accession Q8TF68 downloaded
AlphaFold 

### 3.5 - Extract features present in residues in proximity to the cysteines

In [21]:
def listAppend(featureType, description, begin, end=None):
    ### Append the description and position of a feature to a cysteine's list
    global feature_lists, feature_counts
    if end:
        feature_lists[featureType].append(f'{description} ({begin}-{end})')
    else:
        feature_lists[featureType].append(f'{description} ({begin})')
    feature_counts[featureType] += 1

In [22]:
def isoform(index):
    ### Extract the chosen canonical isoform of a protein that has alternative splicing
    for comment in EBI_output["EBI_data"][index]["comments"]:
        if comment["type"] == "ALTERNATIVE_PRODUCTS":
            for isoform in comment["isoforms"]:
                if isoform["sequenceStatus"] == "displayed":
                    return isoform["ids"]

In [23]:
AlphaFoldModelIDS = []
domains = []
ptms = []
feature_counts = {"domains": 0, "ptms": 0}

for i in range(len(AlphaFold_output)):
    proximity_residues = AlphaFold_output["proximity_residues"][i]
    feature_lists = {"domains": [], "ptms": []}

    # Loop over domains and PTMs of the protein, select those that are spatially close to the cysteine
    for j in EBI_output["EBI_data"][i]["features"]:
        try:
            begin = int(j["begin"])
        except:  # when beginning of a feature is "?"
            begin = 0
        try:
            end = int(j["end"])
        except:  # when end of a feature is "?"
            end = 9999        
        if j["molecule"] and j["molecule"] != isoform(i):
            break # if feature is in an isoform that is not the canonical

        if j["category"] == "DOMAINS_AND_SITES":
            # Binding sites in proximity to the cysteine
            if j["type"] in ["BINDING", "SITE", "ACT_SITE"]:
                if begin in proximity_residues.keys():
                    if j["type"] == "BINDING":
                        description = f'{j["ligand"]["name"]} binding site'
                    elif j["type"] == "ACT_SITE" and not j["description"]:
                        description = "Active site"
                    else:
                        description = j["description"]
                    listAppend("domains", description, f'{begin}, {proximity_residues[begin]:.2f} Å')
            # Domains that include residues close to the cysteine
            else:
                if any(x in proximity_residues.keys() for x in list(range(begin, end+1))):
                    residues_in_domain = {residue: distance for residue, distance in proximity_residues.items() if begin <= residue <= end}
                    closest_residue = min(residues_in_domain, key=residues_in_domain.get)
                    try:
                        if j["type"] == "DNA_BIND" and not j["description"]:
                            listAppend("domains", "DNA binding", begin, f'{end}) - ({closest_residue}, {proximity_residues[closest_residue]:.2f} Å')
                        else:
                            listAppend("domains", j["description"], begin, f'{end}) - ({closest_residue}, {proximity_residues[closest_residue]:.2f} Å')
                    except:  # for those domains that don't have a description
                        if j["type"] == "COILED":
                            listAppend("domains", "Coiled coil", begin, f'{end}) - ({closest_residue}, {proximity_residues[closest_residue]:.2f} Å')

        if j["category"] == "PTM":
            # Disulfide bonds
            if j["type"] == "DISULFID":
                if begin in proximity_residues.keys():
                    listAppend("ptms", "Disulfide bond", f'{begin}, {proximity_residues[begin]:.2f} Å ', f' {end}')
                elif end in proximity_residues.keys():
                    listAppend("ptms", "Disulfide bond", f'{begin} ', f' {end}, {proximity_residues[end]:.2f} Å')
            # PTMs in proximity to the cysteine
            else:
                if begin in proximity_residues.keys():
                    ptm_name = j["description"].split(";")[0]
                    listAppend("ptms", ptm_name, f'{begin}, {proximity_residues[begin]:.2f} Å')

    AlphaFoldModelIDS.append(AlphaFold_output["AlphaFold_model_ID"][i])
    domains.append(feature_lists["domains"])
    ptms.append(feature_lists["ptms"])

AlphaFold_features = pd.DataFrame({"AlphaFold_model_ID": AlphaFoldModelIDS, "nearby_domains": domains, "nearby_PTMs": ptms})
print(f'{feature_counts["domains"]} nearby domains and {feature_counts["ptms"]} nearby PTMs extracted in total')

4613 nearby domains and 1881 nearby PTMs extracted in total


### 3.6 - Create a DataFrame with all the new features appended to the input data and save it

In [24]:
final_output = pd.concat([input_data, AIUPred_features, EBI_features, AlphaFold_features], axis=1)

# Change the order of columns
cols = list(final_output.columns)
new_cols = [cols[-12], cols[-11], cols[-5], cols[-3], cols[-4], cols[-6], cols[-10], cols[-9], cols[-8], cols[-7], cols[-2], cols[-1]]
final_output = final_output[new_cols]

final_output.to_excel("Analysis_output.xlsx")
final_output

,protein accession number,gene name,full_name,AlphaFold_model_ID,subcellular_loc,protein_length,cysteine site,sensitive,cysteine_within,disorder_value,nearby_domains,nearby_PTMs
0,P49588,AARS1,"Alanine--tRNA ligase, cytoplasmic",AF-P49588-F1,"[Cytoplasm, Nucleus]",968,C773,No,No,0.16,"[Nuclear localization signal (750-763) - (763,...",[]
1,P49588,AARS1,"Alanine--tRNA ligase, cytoplasmic",AF-P49588-F1,"[Cytoplasm, Nucleus]",968,C903,No,No,0.13,[],"[N6-acetyllysine (876, 10.35 Å), N6,N6,N6-trim..."
2,P33527,ABCC1,ATP-binding cassette sub-family C member 1,AF-P33527-F1,"[Cell membrane, Basolateral cell membrane]",1531,C744,No,No,0.05,"[ABC transporter 1 (644-868) - (744, 0.00 Å), ...",[]
3,Q8NE71,ABCF1,ATP-binding cassette sub-family F member 1,AF-Q8NE71-F1,"[Cytoplasm, Nucleus, nucleoplasm, Nucleus enve...",845,C655,No,No,0.06,"[ABC transporter 1 (304-548) - (484, 10.64 Å),...",[]
4,Q9NUJ1,ABHD10,"Palmitoyl-protein thioesterase ABHD10, mitocho...",AF-Q9NUJ1-F1,[Mitochondrion],306,C100,No,No,0.03,"[AB hydrolase-1 (78-178) - (100, 0.00 Å)]",[]
...,...,...,...,...,...,...,...,...,...,...,...,...
1719,Q96KR1,ZFR,Zinc finger RNA-binding protein,AF-Q96KR1-F1,"[Nucleus, Cytoplasm, Cytoplasmic granule, Chro...",1074,C922,Yes,No,0.04,"[DZF (703-1073) - (922, 0.00 Å), Disordered (7...",[]
1720,Q8N5A5,ZGPAT,Zinc finger CCCH-type with G patch domain-cont...,AF-Q8N5A5-F1,"[Nucleus, Nucleus]",531,C377,Yes,Yes,0.35,"[G-patch (333-379) - (377, 0.00 Å), Disordered...","[Phosphoserine (373, 6.34 Å)]"
1721,Q8TF68,ZNF384,Zinc finger protein 384,AF-Q8TF68-F1,[Nucleus],577,C435,Yes,Yes,0.26,"[C2H2-type 7 (403-425) - (425, 18.81 Å), C2H2-...",[]
1722,Q2TB10,ZNF800,Zinc finger protein 800,AF-Q2TB10-F1,"[Nucleus, Chromosome]",664,C359,Yes,Yes,0.37,"[C2H2-type 3 (287-310) - (310, 15.94 Å), C2H2-...",[]


### 3.7 - A few considerations on the findings from scraping data from AIUPred.

In [25]:
print(f'Out of the {len(np.unique(final_output[accession_column]))} proteins in the data set, \
{len(np.unique(final_output[final_output["sensitive"] == "Yes"][accession_column]))} are sensitive to oxidation, \
with a total of {final_output[final_output["sensitive"] == "Yes"].shape[0]} cysteines in them.\n\
Of the {final_output.shape[0]} total cysteines in the data set, \
{final_output[final_output["cysteine_within"] == "Yes"].shape[0]} lie in a region affected by the oxidative environment.')

Out of the 1166 proteins in the data set, 408 are sensitive to oxidation, with a total of 643 cysteines in them.
Of the 1724 total cysteines in the data set, 148 lie in a region affected by the oxidative environment.
